# Tutorial 4: Agents in LangChain

In this tutorial, we'll explore Agents in LangChain, which are autonomous entities capable of using tools and making decisions to accomplish tasks.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

load_dotenv()

llm = ChatGroq(model_name='qwen/qwen3-32b', temperature=0.1)
print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 1. Understanding Agent Architecture

Agents in LangChain combine an LLM with a set of tools to perform tasks. The LLM decides which tools to use and how to interpret their outputs.

## 2. Exploring Different Types of Agents

### Zero-shot React Agent

In [2]:
search = DuckDuckGoSearchRun()

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    try:
        return search.run(query)[:500]
    except Exception as e:
        return f"Search unavailable: {e}"

# create_react_agent replaces initialize_agent + AgentType
tools = [get_word_length, web_search]
zero_shot_agent = create_react_agent(model=llm, tools=tools)

result = zero_shot_agent.invoke({
    "messages": [HumanMessage(content="What is the square root of the year Pythagoras was born? Search for the year first.")]
})
print(result["messages"][-1].content[:300])

The square root of the year Pythagoras was born (approximately 570 BC) is calculated by taking the square root of 570. 

$$
\sqrt{570} \approx 23.87
$$

Note: Since 570 BC is represented as -570 in historical timelines, the square root of a negative number isn't real. However, treating the year as a


In [3]:
# Web search tool is already defined above as web_search
# Tool from langchain_community.tools for backward compatibility:
from langchain_community.tools import Tool
web_search_tool = Tool(
    name='DuckDuckGoSearch',
    func=search.run,
    description='Search the web for current information'
)
print("Web search tool defined:", web_search_tool.name)

Web search tool defined: DuckDuckGoSearch


### Conversational Agent

In [5]:
# Conversational agent with chat history — LangGraph maintains message history automatically
conversational_agent = create_react_agent(
    model=llm,
    tools=[web_search, get_word_length],
    prompt='You are a helpful conversational assistant. Answer questions and remember context.'
)

result = conversational_agent.invoke({
    "messages": [HumanMessage(content="Hello! I'd like to learn about the theory of relativity. Can you summarize it briefly?")]
})
print(result["messages"][-1].content[:300])

The theory of relativity, developed by Albert Einstein, consists of two parts:

1. **Special Relativity (1905)**:  
   - **Key Postulates**:  
     - The laws of physics are the same in all inertial (non-accelerating) frames of reference.  
     - The speed of light in a vacuum is constant for all o


### STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION Agent

In [6]:
# Plan-and-execute style agent — prompt instructs the model to plan before acting
plan_execute_agent = create_react_agent(
    model=llm,
    tools=[web_search, get_word_length],
    prompt=(
        'You are a careful problem-solver. Before acting:\n'
        '1. Break the problem into steps\n'
        '2. Execute each step using available tools\n'
        '3. Combine results into a final answer'
    )
)

result = plan_execute_agent.invoke({
    "messages": [HumanMessage(content="Find out who invented the telephone and calculate the number of years between that invention and the first moon landing in 1969.")]
})
print(result["messages"][-1].content[:300])

The telephone was invented by Alexander Graham Bell, who was awarded the first U.S. patent for the invention in 1876. The first moon landing occurred in 1969. 

To calculate the number of years between these events:  
1969 (moon landing) − 1876 (telephone invention) = **93 years**.  

**Answer:** Th


## 3. Creating Custom Tools for Agents

In [7]:
@tool
def get_word_length_tool(word: str) -> int:
    """Returns the number of characters in a word."""
    return len(word)

custom_tools = [web_search, get_word_length_tool]
custom_agent = create_react_agent(
    model=llm,
    tools=custom_tools,
    prompt='You are a versatile assistant with access to web search and word length tools.'
)

result = custom_agent.invoke({
    "messages": [HumanMessage(content="How many letters are in the name of the inventor of the World Wide Web?")]
})
print(result["messages"][-1].content[:200])

The name of the inventor of the World Wide Web, Tim Berners-Lee, has **13 letters** when combined without spaces or hyphens. 

Answer: 13.


## 4. Implementing a Multi-tool Agent for Task Solving

In [8]:
# Multi-tool agent using all available tools
all_tools = [web_search, get_word_length]
multi_tool_agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=(
        'You are a multi-capable assistant. Use available tools to:\n'
        '1. Find the current temperature in New York City\n'
        '2. Calculate the square root of that temperature\n'
        '3. Report both results clearly'
    )
)

result = multi_tool_agent.invoke({
    "messages": [HumanMessage(content="Find the current temperature in New York City and calculate its square root.")]
})
print(result["messages"][-1].content[:300])

The current temperature in New York City is **26°C**. The square root of this temperature is approximately **5.10**.


## Conclusion

In this tutorial, we've explored Agents in LangChain, including different types of agents, creating custom tools, and implementing a multi-tool agent for complex task solving. Agents provide a powerful way to create autonomous systems that can leverage various tools and make decisions to accomplish tasks.